# Taller 2 - Punto 1: Embeddings de Palabras con Word2vec y FastText

Este notebook implementa el entrenamiento y evaluación de modelos de embeddings de palabras usando Word2vec y FastText sobre el dataset spanish_billion_words.

## Estrategia de Experimentación:
- Dataset completo (46.9M textos) y muestra de 500k
- Modelos: Word2Vec y FastText
- Dimensiones de embeddings: 100, 200, 300
- Total de experimentos: 12 (2 modelos x 3 dimensiones x 2 tamaños de dataset)
- Resultados guardados en JSON para análisis posterior
- Visualizaciones y modelos organizados en directorios separados

## 1.1 Instalación de Dependencias y Configuración Inicial

In [ ]:
import nltk
nltk.download('stopwords')

from datasets import load_dataset
from nltk.corpus import stopwords
from tqdm.auto import tqdm
import re
import string
from gensim.models import FastText, Word2Vec
import numpy as np
import warnings
import time
import gc
warnings.filterwarnings('ignore')

from helpers import (
    save_model, 
    save_experiment_results, 
    load_experiment_results,
    visualize_embeddings_tsne,
    visualize_embeddings_pca,
    query_similar_words,
    print_experiment_summary,
    clear_model_from_memory,
    ensure_directories
)

ensure_directories(1)

In [ ]:
EJECUTAR_TODOS = True
PROCESAR_COMPLETO = True

## 1.2 Carga del Dataset Spanish Billion Words

Cargamos el dataset completo y preparamos las muestras para experimentación.

In [ ]:
import os

CACHE_DIR = "./input/jhonparra18"
os.makedirs(CACHE_DIR, exist_ok=True)

print("Cargando dataset spanish_billion_words...")
dataset_full = load_dataset(
    "jhonparra18/spanish_billion_words_clean", 
    split="train",
    cache_dir=CACHE_DIR
)

print(f"Dataset completo cargado: {len(dataset_full):,} textos")
print(f"Cache guardado en: {CACHE_DIR}")

## 1.3 Preprocesamiento del Corpus

Función de limpieza del texto que se aplicará a cada muestra del dataset.

In [ ]:
def preprocess_text(dataset_sample):
    """
    Preprocesa el texto eliminando números, stopwords, puntuación y símbolos especiales
    """
    sentences = []
    stop_words = stopwords.words('spanish')
    
    print(f"Procesando {len(dataset_sample['text']):,} textos...")
    
    for sent in tqdm(dataset_sample['text'], desc="Limpiando texto"):
        if len(sent) > 1:
            words = sent.split()
            words = [w for w in words if not w.isdigit()]
            words = [re.sub(r'[0-9]', '', w) for w in words]
            words = [w for w in words if w.lower() not in stop_words]
            
            re_punc = re.compile('[%s]' % re.escape(string.punctuation))
            words = [re_punc.sub('', w) for w in words]
            words = [re.sub(r"\!|\'|\?|\¿|\¡|\«|\»", "", w) for w in words]
            words = [w.lower() for w in words if w != '']
            
            if len(words) > 0:
                sentences.append(words)
    
    print(f"Total de oraciones procesadas: {len(sentences):,}")
    return sentences

## 1.3.1 Preprocesamiento de Datasets

Preprocesamos ambos datasets (500k y completo) una sola vez para reutilizarlos en todos los experimentos.

In [ ]:
print("="*80)
print("PREPROCESANDO DATASETS")
print("="*80)

print("\n1. Preparando muestra de 500,000 textos...")
dataset_500k = dataset_full.select(range(min(500000, len(dataset_full))))
sentences_500k = preprocess_text(dataset_500k)

print("\n2. Preparando dataset completo...")
print("ADVERTENCIA: Esto tomará mucho tiempo y memoria.")
print("Para pruebas, considere ejecutar solo experimentos con 500k.")

if PROCESAR_COMPLETO:
    sentences_full = preprocess_text(dataset_full)
    print(f"\nDataset completo procesado: {len(sentences_full):,} sentencias")
else:
    sentences_full = None
    print("\nDataset completo NO procesado (PROCESAR_COMPLETO = False)")

print("\n" + "="*80)
print("DATASETS PREPROCESADOS Y LISTOS")
print("="*80)
print(f"  500k: {len(sentences_500k):,} sentencias")
print(f"  Full: {'Procesado' if sentences_full else 'No procesado'}")
print("="*80)

## 1.4 Configuración de Experimentos

Definimos todos los experimentos a ejecutar con diferentes combinaciones de modelos, dimensiones y tamaños de dataset.

In [ ]:
experiments = [
    {'model_type': 'word2vec', 'vector_size': 100, 'dataset_size': 500000, 'dataset_name': '500k'},
    {'model_type': 'word2vec', 'vector_size': 200, 'dataset_size': 500000, 'dataset_name': '500k'},
    {'model_type': 'word2vec', 'vector_size': 300, 'dataset_size': 500000, 'dataset_name': '500k'},
    {'model_type': 'word2vec', 'vector_size': 100, 'dataset_size': None, 'dataset_name': 'full'},
    {'model_type': 'word2vec', 'vector_size': 200, 'dataset_size': None, 'dataset_name': 'full'},
    {'model_type': 'word2vec', 'vector_size': 300, 'dataset_size': None, 'dataset_name': 'full'},
    
    {'model_type': 'fasttext', 'vector_size': 100, 'dataset_size': 500000, 'dataset_name': '500k'},
    {'model_type': 'fasttext', 'vector_size': 200, 'dataset_size': 500000, 'dataset_name': '500k'},
    {'model_type': 'fasttext', 'vector_size': 300, 'dataset_size': 500000, 'dataset_name': '500k'},
    {'model_type': 'fasttext', 'vector_size': 100, 'dataset_size': None, 'dataset_name': 'full'},
    {'model_type': 'fasttext', 'vector_size': 200, 'dataset_size': None, 'dataset_name': 'full'},
    {'model_type': 'fasttext', 'vector_size': 300, 'dataset_size': None, 'dataset_name': 'full'},
]

print(f"Total de experimentos configurados: {len(experiments)}")
for i, exp in enumerate(experiments, 1):
    print(f"{i:2}. {exp['model_type']:10} - {exp['vector_size']}d - {exp['dataset_name']:5}")

## 1.5 Función de Entrenamiento y Evaluación

Función unificada que ejecuta un experimento completo: entrenamiento, evaluación, visualización y guardado de resultados.
OPTIMIZACIÓN: Usa datasets ya preprocesados en lugar de preprocesar cada vez.

In [ ]:
def run_experiment(exp_config, sentences_500k, sentences_full):
    """
    Ejecuta un experimento completo de entrenamiento y evaluación
    Usa datasets ya preprocesados para evitar reprocesamiento
    """
    model_type = exp_config['model_type']
    vector_size = exp_config['vector_size']
    dataset_size = exp_config['dataset_size']
    dataset_name = exp_config['dataset_name']
    
    experiment_name = f"{model_type}_{vector_size}d_{dataset_name}"
    
    print("\n" + "="*80)
    print(f"EXPERIMENTO: {experiment_name}")
    print("="*80)
    
    if dataset_size:
        sentences = sentences_500k
        print(f"Usando muestra 500k: {len(sentences):,} sentencias preprocesadas")
    else:
        if sentences_full is None:
            print("ERROR: Dataset completo no está preprocesado.")
            print("Cambie PROCESAR_COMPLETO = True en la celda de preprocesamiento.")
            return None
        sentences = sentences_full
        print(f"Usando dataset completo: {len(sentences):,} sentencias preprocesadas")
    
    print(f"\nEntrenando modelo {model_type.upper()} con vector_size={vector_size}...")
    start_time = time.time()
    
    if model_type == 'word2vec':
        model = Word2Vec(
            sentences=sentences,
            vector_size=vector_size,
            window=5,
            min_count=10,
            workers=4,
            sg=0
        )
    else:
        model = FastText(
            sentences=sentences,
            vector_size=vector_size,
            window=5,
            min_count=10,
            workers=4,
            sg=0
        )
    
    training_time = time.time() - start_time
    vocab_size = len(model.wv.index_to_key)
    
    print(f"Entrenamiento completado en {training_time:.2f} segundos")
    print(f"Vocabulario: {vocab_size:,} palabras")
    
    model_path = save_model(model, experiment_name, punto=1)
    print(f"Modelo guardado: {model_path}")
    
    test_word = 'futuro'
    print(f"\nConsultando similitud para '{test_word}'...")
    similar_words_result = query_similar_words(model, test_word, topn=5)
    
    if 'similar_words' in similar_words_result:
        for sw in similar_words_result['similar_words']:
            print(f"  {sw['word']}: {sw['similarity']:.4f}")
    
    print(f"\nGenerando visualización t-SNE...")
    tsne_path = visualize_embeddings_tsne(model, experiment_name, punto=1, num_words=100)
    
    print(f"\nGenerando visualización PCA...")
    pca_path, variance_explained = visualize_embeddings_pca(model, experiment_name, punto=1, num_words=100)
    
    results = {
        'model_type': model_type,
        'vector_size': vector_size,
        'dataset_name': dataset_name,
        'dataset_size': len(sentences),
        'vocab_size': vocab_size,
        'training_time': training_time,
        'variance_explained': variance_explained,
        'model_path': model_path,
        'visualizations': [tsne_path, pca_path],
        'similar_words_sample': similar_words_result
    }
    
    results_file = save_experiment_results(punto=1, experiment_name=experiment_name, results=results)
    print(f"\nResultados guardados en: {results_file}")
    
    print(f"\nLiberando memoria del modelo...")
    clear_model_from_memory(model)
    gc.collect()
    
    print(f"\nEXPERIMENTO {experiment_name} COMPLETADO")
    print("="*80)
    
    return results

## 1.6 Ejecución de Experimentos

Ejecutamos todos los experimentos configurados usando los datasets ya preprocesados.

In [ ]:
if EJECUTAR_TODOS:
    experimentos_a_ejecutar = experiments
else:
    experimentos_a_ejecutar = [exp for exp in experiments if exp['dataset_name'] == '500k'][:3]

print(f"Se ejecutarán {len(experimentos_a_ejecutar)} experimentos")
if not EJECUTAR_TODOS:
    print("Para ejecutar todos los experimentos (incluyendo dataset completo):")
    print("  1. Cambie PROCESAR_COMPLETO = True en la celda 1.3.1")
    print("  2. Cambie EJECUTAR_TODOS = True aquí")
print()

for i, exp in enumerate(experimentos_a_ejecutar, 1):
    print(f"\n{'#'*80}")
    print(f"# EXPERIMENTO {i}/{len(experimentos_a_ejecutar)}")
    print(f"{'#'*80}")
    
    try:
        result = run_experiment(exp, sentences_500k, sentences_full)
        if result is None:
            print("Experimento omitido (dataset no disponible)")
            continue
    except Exception as e:
        print(f"\nERROR en experimento {exp}: {e}")
        import traceback
        traceback.print_exc()
        continue

print("\n" + "="*80)
print("TODOS LOS EXPERIMENTOS COMPLETADOS")
print("="*80)

## 1.7 Visualización de Resultados

Imprimimos un resumen de todos los experimentos ejecutados desde el archivo JSON.

In [ ]:
print_experiment_summary(punto=1)

## 1.8 Análisis Comparativo Detallado

Análisis estadístico y comparativo de los resultados obtenidos.

In [ ]:
import pandas as pd

results = load_experiment_results(punto=1)

if results:
    df = pd.DataFrame(results)
    
    print("\n" + "="*80)
    print("ANÁLISIS COMPARATIVO DE EXPERIMENTOS")
    print("="*80)
    
    print("\n1. TIEMPOS DE ENTRENAMIENTO:")
    for model_type in df['model_type'].unique():
        subset = df[df['model_type'] == model_type]
        print(f"\n   {model_type.upper()}:")
        for _, row in subset.iterrows():
            print(f"     {row['vector_size']}d - {row['dataset_name']:5}: {row['training_time']:7.2f}s")
    
    print("\n2. VARIANZA EXPLICADA (PCA):")
    for model_type in df['model_type'].unique():
        subset = df[df['model_type'] == model_type]
        print(f"\n   {model_type.upper()}:")
        for _, row in subset.iterrows():
            print(f"     {row['vector_size']}d - {row['dataset_name']:5}: {row['variance_explained']*100:5.2f}%")
    
    print("\n3. TAMAÑOS DE VOCABULARIO:")
    for model_type in df['model_type'].unique():
        subset = df[df['model_type'] == model_type]
        print(f"\n   {model_type.upper()}:")
        for _, row in subset.iterrows():
            print(f"     {row['vector_size']}d - {row['dataset_name']:5}: {row['vocab_size']:,} palabras")
    
    print("\n4. COMPARACIÓN WORD2VEC vs FASTTEXT:")
    if len(df[df['model_type'] == 'word2vec']) > 0 and len(df[df['model_type'] == 'fasttext']) > 0:
        w2v_avg_time = df[df['model_type'] == 'word2vec']['training_time'].mean()
        ft_avg_time = df[df['model_type'] == 'fasttext']['training_time'].mean()
        w2v_avg_var = df[df['model_type'] == 'word2vec']['variance_explained'].mean()
        ft_avg_var = df[df['model_type'] == 'fasttext']['variance_explained'].mean()
        
        print(f"\n   Tiempo promedio de entrenamiento:")
        print(f"     Word2Vec: {w2v_avg_time:.2f}s")
        print(f"     FastText: {ft_avg_time:.2f}s")
        print(f"\n   Varianza explicada promedio (PCA):")
        print(f"     Word2Vec: {w2v_avg_var*100:.2f}%")
        print(f"     FastText: {ft_avg_var*100:.2f}%")
    
    print("\n5. MEJOR CONFIGURACIÓN POR MÉTRICA:")
    best_variance = df.loc[df['variance_explained'].idxmax()]
    fastest = df.loc[df['training_time'].idxmin()]
    largest_vocab = df.loc[df['vocab_size'].idxmax()]
    
    print(f"\n   Mayor varianza explicada:")
    print(f"     {best_variance['experiment_name']}: {best_variance['variance_explained']*100:.2f}%")
    print(f"\n   Entrenamiento más rápido:")
    print(f"     {fastest['experiment_name']}: {fastest['training_time']:.2f}s")
    print(f"\n   Mayor vocabulario:")
    print(f"     {largest_vocab['experiment_name']}: {largest_vocab['vocab_size']:,} palabras")
    
    print("\n" + "="*80)
else:
    print("No hay resultados disponibles. Ejecute los experimentos primero.")

## 1.9 Conclusiones

Conclusiones finales basadas en los experimentos ejecutados.

In [ ]:
results = load_experiment_results(punto=1)

print("\n" + "="*80)
print("CONCLUSIONES - PUNTO 1: EMBEDDINGS DE PALABRAS")
print("="*80)

if results:
    df = pd.DataFrame(results)
    
    print(f"\n1. EXPERIMENTOS REALIZADOS:")
    print(f"   Total de experimentos: {len(results)}")
    print(f"   Modelos evaluados: {', '.join(df['model_type'].unique())}")
    print(f"   Dimensiones probadas: {sorted(df['vector_size'].unique())}")
    print(f"   Tamaños de dataset: {', '.join(df['dataset_name'].unique())}")
    
    print(f"\n2. CORPUS Y PREPROCESAMIENTO:")
    max_dataset = df['dataset_size'].max()
    min_dataset = df['dataset_size'].min()
    print(f"   Dataset completo: {max_dataset:,} sentencias procesadas")
    print(f"   Dataset muestra: {min_dataset:,} sentencias procesadas")
    avg_vocab = df['vocab_size'].mean()
    print(f"   Vocabulario promedio: {avg_vocab:,.0f} palabras únicas")
    
    print(f"\n3. RENDIMIENTO:")
    print(f"   Tiempo de entrenamiento (promedio): {df['training_time'].mean():.2f}s")
    print(f"   Tiempo mínimo: {df['training_time'].min():.2f}s")
    print(f"   Tiempo máximo: {df['training_time'].max():.2f}s")
    
    print(f"\n4. CALIDAD DE EMBEDDINGS:")
    print(f"   Varianza explicada (PCA promedio): {df['variance_explained'].mean()*100:.2f}%")
    print(f"   Mejor varianza explicada: {df['variance_explained'].max()*100:.2f}%")
    
    if 'word2vec' in df['model_type'].values and 'fasttext' in df['model_type'].values:
        w2v_var = df[df['model_type'] == 'word2vec']['variance_explained'].mean()
        ft_var = df[df['model_type'] == 'fasttext']['variance_explained'].mean()
        
        print(f"\n5. COMPARACIÓN WORD2VEC vs FASTTEXT:")
        print(f"   Word2Vec:")
        print(f"     - Varianza promedio: {w2v_var*100:.2f}%")
        print(f"     - Mejor para: Relaciones semánticas puras")
        print(f"   FastText:")
        print(f"     - Varianza promedio: {ft_var*100:.2f}%")
        print(f"     - Mejor para: Manejo de palabras OOV y variaciones morfológicas")
    
    print(f"\n6. RECOMENDACIONES:")
    best = df.loc[df['variance_explained'].idxmax()]
    print(f"   Configuración óptima: {best['experiment_name']}")
    print(f"   - Tipo: {best['model_type'].upper()}")
    print(f"   - Dimensión: {best['vector_size']}")
    print(f"   - Dataset: {best['dataset_name']}")
    print(f"   - Varianza explicada: {best['variance_explained']*100:.2f}%")
    
    print(f"\n7. ARCHIVOS GENERADOS:")
    print(f"   Modelos guardados en: ./models/punto1/")
    print(f"   Visualizaciones en: ./output/punto1/")
    print(f"   Resultados JSON en: ./results/punto1_results.json")
else:
    print("\nNo hay resultados disponibles.")
    print("Ejecute los experimentos en la sección 1.6 primero.")

print("\n" + "="*80)